In [ ]:
from pathlib import Path
import os
import subprocess

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "train_model.py").exists():
    repo_root = repo_root.parent
if not (repo_root / "src" / "train_model.py").exists():
    repo_root = Path("/content/09_12523118_10123078_CustomerChurn")
    if not repo_root.exists():
        subprocess.run(["git", "clone", "https://github.com/TaDucHuy0810/09_12523118_10123078_CustomerChurn.git", str(repo_root)], check=True)
os.chdir(repo_root)
print("Project root:", repo_root)

# 01 - EDA Customer Churn

Notebook dùng dataset `data/telco_churn.csv` trong repository. Nếu chạy trên Colab, clone repository trước rồi chuyển working directory vào thư mục gốc. Cell code bên dưới chạy `notebooks/eda.py` và hiển thị bảy hình đã lưu ở `docs/figures/`.

Mỗi hình được phân tích theo ba ý: quan sát có số liệu, ý nghĩa với bài toán churn, và quyết định xử lý tiếp theo. Tỷ lệ bên dưới tính trên 7.043 dòng gốc.

## Phân bố nhãn

1. **Hình cho thấy gì:** 5.174 khách hàng không churn (73,46%), 1.869 churn (26,54%).
2. **Ý nghĩa:** nhãn churn chiếm thiểu số, nên Accuracy đơn lẻ có thể che khuất lỗi bỏ sót khách churn.
3. **Quyết định:** dùng stratified split và báo cáo Precision, Recall, F1, ROC-AUC.

## Churn theo giới tính

1. **Hình cho thấy gì:** tỷ lệ churn nữ 26,92%, nam 26,16%.
2. **Ý nghĩa:** chênh lệch nhỏ trong mẫu hiện tại.
3. **Quyết định:** giữ Gender nhưng không diễn giải là nguyên nhân churn.

## Churn theo hợp đồng

1. **Hình cho thấy gì:** month-to-month 42,71%, one-year 11,27%, two-year 2,83%.
2. **Ý nghĩa:** hợp đồng ngắn hạn đi cùng tỷ lệ churn cao hơn.
3. **Quyết định:** giữ Contract dạng phân loại, One-Hot encode; đây là liên hệ quan sát được, không phải kết luận nhân quả.

## Thời gian sử dụng

1. **Hình cho thấy gì:** median tenure nhóm churn là 10 tháng, nhóm không churn là 38 tháng.
2. **Ý nghĩa:** khách hàng mới có xu hướng xuất hiện nhiều hơn trong nhóm churn.
3. **Quyết định:** giữ Tenure Months dạng số và scale bên trong pipeline.

## Phí hàng tháng

1. **Hình cho thấy gì:** median phí nhóm churn 79,65, nhóm không churn 64,43.
2. **Ý nghĩa:** phí cao có liên hệ với churn, nhưng có thể đi cùng loại dịch vụ/hợp đồng.
3. **Quyết định:** parse sang số, giữ ngoại lệ hợp lệ và scale.

## Phương thức thanh toán

1. **Hình cho thấy gì:** Electronic check 45,29%; Bank transfer 16,71%; Credit card 15,24%; Mailed check 19,11%.
2. **Ý nghĩa:** Electronic check có tỷ lệ churn cao nhất trong các nhóm thanh toán.
3. **Quyết định:** giữ Payment Method, phân tích cùng Contract/Internet Service.

## Dịch vụ Internet

1. **Hình cho thấy gì:** Fiber optic 41,89%; DSL 18,96%; không dùng Internet 7,40%.
2. **Ý nghĩa:** tỷ lệ churn khác nhau giữa các nhóm dịch vụ.
3. **Quyết định:** giữ Internet Service dạng phân loại; không suy diễn quan hệ nhân quả.

In [ ]:
from pathlib import Path
import runpy
import sys
from IPython.display import Image, display

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / 'src' / 'train_model.py').exists():
    project_root = project_root.parent
if not (project_root / 'src' / 'train_model.py').exists():
    raise RuntimeError('Clone the repository and set the working directory to its root before running this notebook.')
sys.path.insert(0, str(project_root))
_ = runpy.run_path(str(project_root / 'notebooks' / 'eda.py'))

figure_names = [
    'eda_churn_distribution.png', 'eda_churn_by_gender.png',
    'eda_churn_by_contract.png', 'eda_tenure_distribution.png',
    'eda_monthly_charges.png', 'eda_churn_by_payment_method.png',
    'eda_churn_by_internet_service.png',
]
for figure_name in figure_names:
    display(Image(filename=str(project_root / 'docs' / 'figures' / figure_name)))